In [ ]:
import itertools
import logging
import os
import pickle
import random
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

## 1 read

In [ ]:
ourdata = pd.read_csv(
    our_data + "fasta_inchi_plant_add.txt",
    sep="\t",
    header=None,
    names=["enzyme", "substrate", "sequence", "inchi"],
)
ourdata = ourdata.sample(frac=1, random_state=42)

## 2 generate

In [ ]:
# ! python extract.py esm1b_t33_650M_UR50S {our_data}our_data_add_q.fasta {our_data}esm --repr_layers 33 --include mean

In [ ]:
ourdata["ESM1b"] = ""
ourdata["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "esm/")

for ind in ourdata.index:
    esms = torch.load(rep_dict + str(ourdata["enzyme"][ind]) + ".pt")
    ecfps = Chem.MolFromInchi(ourdata["inchi"][ind])
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(
        ecfps, radius=2, nBits=1024
    ).ToBitString()
    ourdata["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    ourdata["ECFP"][ind] = ecfpso

In [ ]:
ourdata["Binding"] = 1

In [ ]:
has_null_values = ourdata.isna().any().any()
has_empty_strings = ourdata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

## 3 convert

In [ ]:
enzyme_unique_count = ourdata["enzyme"].nunique()
substrate_unique_count = ourdata["substrate"].nunique()
inchi_unique_count = ourdata["inchi"].nunique()
ecfp_unique_count = ourdata["ECFP"].nunique()

In [ ]:
num_partitions = 5
partition_size = len(ourdata) // num_partitions
random.seed(42)
all_substrate = ourdata["substrate"].unique()
all_enzyme = np.array([])

ourdatas = [
    ourdata.iloc[i * partition_size : (i + 1) * partition_size]
    for i in range(num_partitions)
]
ourdata_negs = []
for num, ourdata_i in enumerate(ourdatas):
    # ----------------------------------------------------------------------------------------------
    enzyme_unique = ourdata_i["enzyme"].unique()
    ourdata_neg_i = pd.DataFrame(
        columns=["enzyme", "substrate", "sequence", "inchi", "ESM1b"]
    )
    rows_to_add = []
    for enzyme in enzyme_unique:
        selected_substrates = random.sample(list(all_substrate), 2)
        rows_to_add.append({"enzyme": enzyme, "substrate": selected_substrates[0]})
        rows_to_add.append({"enzyme": enzyme, "substrate": selected_substrates[1]})
    ourdata_neg_i = pd.concat(
        [ourdata_neg_i, pd.DataFrame(rows_to_add)], ignore_index=True
    )
    for index, row in ourdata_i.iterrows():
        matching_rows = ourdata_neg_i[(ourdata_neg_i["enzyme"] == row["enzyme"])]
        if not matching_rows.empty:
            for matching_index in matching_rows.index:
                ourdata_neg_i.at[matching_index, "ESM1b"] = row["ESM1b"]

    for index, row in ourdata.iterrows():
        matching_rows = ourdata_neg_i[(ourdata_neg_i["substrate"] == row["substrate"])]
        if not matching_rows.empty:
            for matching_index in matching_rows.index:
                ourdata_neg_i.at[matching_index, "ECFP"] = row["ECFP"]
    ourdata_neg_i["Binding"] = 0
    ourdata_negs.append(ourdata_neg_i)

    merged_data = pd.concat([ourdata_i, ourdata_neg_i], ignore_index=True)
    selected_columns = ["enzyme", "substrate", "Binding", "ECFP", "ESM1b"]
    final_data = merged_data.loc[:, selected_columns]

    final_data.to_pickle(our_data + "p450plant" + str(num) + ".pkl")

    # ----------------------------------------------------------------------------------------------

    slice_enzyme = ourdata_i["enzyme"].unique()

    slice_enzyme = np.setdiff1d(slice_enzyme, all_enzyme)

    for enzyme_j in slice_enzyme:
        if enzyme_j not in all_enzyme:
            all_enzyme = np.append(all_enzyme, enzyme_j)

    combinations = list(itertools.product(slice_enzyme, all_substrate))
    result_df = pd.DataFrame(combinations, columns=["enzyme", "substrate"])
    for index, row in result_df.iterrows():
        the_enzyme = row["enzyme"]
        the_substrate = row["substrate"]
        matching_row = ourdata_i[
            (ourdata_i["enzyme"] == the_enzyme)
            & (ourdata_i["substrate"] == the_substrate)
            & (ourdata_i["Binding"] == 1)
        ]
        if not matching_row.empty:
            result_df.at[index, "Binding"] = 1
        else:
            result_df.at[index, "Binding"] = 0

    result_df["ESM1b"] = None
    result_df["ECFP"] = None
    for index, row in ourdata_i.iterrows():
        matching_rows = result_df[(result_df["enzyme"] == row["enzyme"])]
        if not matching_rows.empty:
            for matching_index in matching_rows.index:
                result_df.at[matching_index, "ESM1b"] = row["ESM1b"]
    for index, row in ourdata.iterrows():
        matching_rows = result_df[(result_df["substrate"] == row["substrate"])]
        if not matching_rows.empty:
            for matching_index in matching_rows.index:
                result_df.at[matching_index, "ECFP"] = row["ECFP"]

    result_df.to_pickle(our_data + "slice" + str(num) + "data.pkl")

In [ ]:
ourdata[ourdata["enzyme"] == "CYP82P3"]

In [ ]:
ourdatas[0][ourdatas[0]["enzyme"] == "CYP82P3"]

In [ ]:
print(ourdatas[1][ourdatas[1]["enzyme"] == "CYP82P3"])
print(ourdatas[2][ourdatas[2]["enzyme"] == "CYP82P3"])
print(ourdatas[3][ourdatas[3]["enzyme"] == "CYP82P3"])
print(ourdatas[4][ourdatas[4]["enzyme"] == "CYP82P3"])

In [ ]:
ourdatas[0]["enzyme"].nunique()

In [ ]:
ourdata["enzyme"].nunique()

In [ ]:
len(ourdata)